# AgentAuth Pocket CA Pipeline

This notebook is the primary runner for the Pocket CA fine-tuning stack.
It assumes you open Jupyter from inside the `agentauth-pocket-ca/` directory
on a Lambda GPU instance.

Before training:

1. `huggingface-cli login`
2. Accept the Llama 3 license for `meta-llama/Meta-Llama-3-8B-Instruct`
3. Confirm the base model in `configs/model.yaml`
4. Decide your run name, for example `pocket-ca-v1`, `pocket-ca-v2`, or `pocket-ca-v3`


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import yaml

# Auto-detect project root: try cwd first, then look for the notebook's directory
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    # Jupyter cwd is not the project dir — search common locations
    candidates = [
        PROJECT_ROOT / "agentauth-pocket-ca",
        Path.home() / "agentauth-pocket-ca",
        Path.home() / "Videos" / "AgentAuth" / "agentauth-pocket-ca",
    ]
    # Also check if any parent of cwd contains the project
    for parent in PROJECT_ROOT.parents:
        candidates.append(parent / "agentauth-pocket-ca")

    found = False
    for candidate in candidates:
        if (candidate / "configs").exists():
            PROJECT_ROOT = candidate.resolve()
            os.chdir(PROJECT_ROOT)
            found = True
            break

    if not found:
        raise RuntimeError(
            "Could not find the agentauth-pocket-ca project directory.\n"
            "Either run: %cd /path/to/agentauth-pocket-ca\n"
            "or set PROJECT_ROOT manually before this cell."
        )
else:
    PROJECT_ROOT = PROJECT_ROOT.resolve()

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
PROJECT_ROOT

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
model_config = yaml.safe_load((PROJECT_ROOT / "configs/model.yaml").read_text())
training_config = yaml.safe_load((PROJECT_ROOT / "configs/training.yaml").read_text())

print("Model config:")
print(json.dumps(model_config, indent=2))
print("\nTraining config:")
print(json.dumps(training_config, indent=2))


## Lambda Run Settings

Set these values before launching the build or training cells.
Leave the dataset paths empty if you only want synthetic data.


In [ ]:
DATASET_SIZE = 100000
RUN_NAME = "pocket-ca-v1"
FORCE_REBUILD_DATASET = False
RESUME_FROM_CHECKPOINT = ""

FINQA_PATH = ""
CONVFINQA_PATH = ""
FINANCEBENCH_PATH = ""
FINR1_PATH = ""

def run_command(command: list[str], extra_env: dict[str, str] | None = None) -> None:
    env = os.environ.copy()
    if extra_env:
        env.update({key: value for key, value in extra_env.items() if value})
    print("$", " ".join(shlex.quote(part) for part in command))
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)

dataset_env = {
    "FINQA_PATH": FINQA_PATH,
    "CONVFINQA_PATH": CONVFINQA_PATH,
    "FINANCEBENCH_PATH": FINANCEBENCH_PATH,
    "FINR1_PATH": FINR1_PATH,
}

print({
    "DATASET_SIZE": DATASET_SIZE,
    "RUN_NAME": RUN_NAME,
    "FORCE_REBUILD_DATASET": FORCE_REBUILD_DATASET,
    "RESUME_FROM_CHECKPOINT": RESUME_FROM_CHECKPOINT or None,
    "DATASET_PATHS": {key: value or None for key, value in dataset_env.items()},
})


## Build Dataset

This generates synthetic financial reasoning samples and prepares the
train, validation, and test splits. If you set any of the dataset path
variables above, those local datasets are merged in during the build.


In [ ]:
run_command(["bash", "scripts/build_dataset.sh", str(DATASET_SIZE)], extra_env=dataset_env)


In [ ]:
sample = json.loads((PROJECT_ROOT / "data/training/train.jsonl").read_text().splitlines()[0])
sample


## Train QLoRA Adapter

This uses the Lambda-friendly launcher so you can change `RUN_NAME`
without editing YAML. Set `RESUME_FROM_CHECKPOINT` to a checkpoint path
if you want to continue an interrupted run.


In [ ]:
train_env = dict(dataset_env)
if FORCE_REBUILD_DATASET:
    train_env["FORCE_REBUILD_DATASET"] = "1"
if RESUME_FROM_CHECKPOINT:
    train_env["RESUME_FROM_CHECKPOINT"] = RESUME_FROM_CHECKPOINT

run_command(["bash", "scripts/launch_training.sh", RUN_NAME, str(DATASET_SIZE)], extra_env=train_env)


## Evaluate

Run both the general reasoning benchmark and the budget-specific benchmark
against the checkpoint for the current `RUN_NAME`.


In [ ]:
checkpoint_path = PROJECT_ROOT / "experiments/checkpoints" / RUN_NAME
run_command(["python", "evaluation/eval_reasoning.py", "--checkpoint", str(checkpoint_path), "--limit", "100"])
run_command(["python", "evaluation/eval_budget.py", "--checkpoint", str(checkpoint_path), "--limit", "100"])


## Serve the Model

For PEFT adapter checkpoints, the API uses Transformers. If you later merge the
adapter into a full checkpoint, set `POCKET_CA_USE_VLLM=1` before running the
server cell and the API will switch to vLLM automatically.


In [ ]:
os.environ["POCKET_CA_MODEL_PATH"] = str(PROJECT_ROOT / "experiments/checkpoints" / RUN_NAME)
print("POCKET_CA_MODEL_PATH=", os.environ["POCKET_CA_MODEL_PATH"])
run_command(["uvicorn", "serving.inference_api:app", "--host", "0.0.0.0", "--port", "8000"])
